In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from conus_biomass import dir_info
from conus_biomass.make_figures import maps

In [ ]:
dir_processed_model_output = dir_info.dir_model_output[:-1] + "_processed/"

In [ ]:
for i in np.arange(0, 50):
    model_suffix = f"_{i:04d}"
    # print(model_suffix)
    df = pd.read_csv(dir_processed_model_output + "MMTC_our_study" + model_suffix + ".csv")
    df = df.drop(columns=["Unnamed: 0"])
    df["model_suffix"] = model_suffix
    if i == 0:
        df_all = df
    else:
        df_all = pd.concat([df_all, df])

In [ ]:
df_all_westwide = df_all.groupby(["year", "model_suffix"]).sum()["live_biomass_MMT"].reset_index()

In [ ]:
# Get the 2005 baseline for each model_suffix
baseline = df_all_westwide[df_all_westwide["year"] == 2005].set_index("model_suffix")[
    "live_biomass_MMT"
]

# Subtract the matching 2005 value based on model_suffix
df_all_westwide["live_biomass_MMT_delta"] = df_all_westwide["live_biomass_MMT"] - df_all_westwide[
    "model_suffix"
].map(baseline)

df_all_westwide["live_biomass_MMT_frac_delta"] = df_all_westwide[
    "live_biomass_MMT"
] / df_all_westwide["model_suffix"].map(baseline)

In [ ]:
df_all_westwide.to_csv("figure_data/figure_2/OurStudy_western_stocks.csv")

In [ ]:
# Get the 2005 baseline for each model_suffix
baseline = df_all_westwide[df_all_westwide["year"] == 2005].set_index("model_suffix")[
    "live_biomass_MMT"
]

# Subtract the matching 2005 value based on model_suffix
df_all_westwide["live_biomass_MMT_delta"] = df_all_westwide["live_biomass_MMT"] - df_all_westwide[
    "model_suffix"
].map(baseline)

df_all_westwide["live_biomass_MMT_frac_delta"] = df_all_westwide[
    "live_biomass_MMT"
] / df_all_westwide["model_suffix"].map(baseline)

In [ ]:
non_CONUS = ["Alaska", "Hawaii", "U.S. Territories", "Puerto Rico"]
western_states = [
    "California",
    "Washington",
    "Oregon",
    "Idaho",
    "Montana",
    "Arizona",
    "Colorado",
    "New Mexico",
    "Utah",
    "Wyoming",
    "Nevada",
]
years = np.arange(2005, 2023)

FRF_state = pd.read_csv(dir_info.dir_walters + "FRF_stock_by_State.csv")

FRF_state_agb = FRF_state[FRF_state["Carbon Pools"] == "Aboveground Biomass"].drop(
    columns=["Carbon Pools"]
)
FRF_state_conus = FRF_state_agb[~FRF_state_agb["State"].isin(non_CONUS)]

CONUS_stocks = FRF_state_conus.drop(columns=["State"]).sum()
CONUS_stocks.index = CONUS_stocks.index.astype(int)

FRF_state_western = FRF_state_conus[FRF_state_conus["State"].isin(western_states)]
western_stocks = FRF_state_western.drop(columns=["State"]).sum()
western_stocks.index = western_stocks.index.astype(int)

FRF_state_eastern = FRF_state_conus[~FRF_state_conus["State"].isin(western_states)]
eastern_stocks = FRF_state_eastern.drop(columns=["State"]).sum()
eastern_stocks.index = eastern_stocks.index.astype(int)

western_stocks_delta = western_stocks - (western_stocks[western_stocks.index == years[0]].values)
eastern_stocks_delta = eastern_stocks - (eastern_stocks[eastern_stocks.index == years[0]].values)
CONUS_stocks_delta = CONUS_stocks - (CONUS_stocks[CONUS_stocks.index == years[0]].values)

USFS_years = CONUS_stocks.index

In [ ]:
western_stocks.to_csv("figure_data/figure_2/USFS_western_stocks.csv")

In [ ]:
df_all_westwide = pd.read_csv("figure_data/figure_2/OurStudy_western_stocks.csv")
western_stocks = pd.read_csv("figure_data/figure_2/USFS_western_stocks.csv")

In [ ]:
western_stocks = western_stocks.rename(columns={"Unnamed: 0": "year", "0": "live_biomass_MMT"})

In [ ]:
biomass_mean_west = df_all_westwide["live_biomass_MMT"].groupby(df_all_westwide["year"]).median()
biomass_min_west = (
    df_all_westwide["live_biomass_MMT"].groupby(df_all_westwide["year"]).quantile(0.025)
)
biomass_max_west = (
    df_all_westwide["live_biomass_MMT"].groupby(df_all_westwide["year"]).quantile(0.975)
)

biomass_mean_west_delta = (
    df_all_westwide["live_biomass_MMT_delta"].groupby(df_all_westwide["year"]).median()
)
biomass_min_west_delta = (
    df_all_westwide["live_biomass_MMT_delta"].groupby(df_all_westwide["year"]).quantile(0.025)
)
biomass_max_west_delta = (
    df_all_westwide["live_biomass_MMT_delta"].groupby(df_all_westwide["year"]).quantile(0.975)
)

# a) absolute biomass stock

In [ ]:
plt.rcParams.update({"font.size": 14})
fig, ax = plt.subplots(figsize=(8, 6))

plt.plot(
    western_stocks["year"],
    western_stocks["live_biomass_MMT"],
    "-",
    linewidth=5,
    color="gray",
    label="USFS/EPA Estimate",
)

plt.plot(
    biomass_mean_west.index,
    (np.array(biomass_mean_west)),
    ".-",
    linewidth=5,
    color="C0",
    label="This Study",
)
plt.fill_between(
    biomass_min_west.index,
    (np.array(biomass_min_west)),
    (np.array(biomass_max_west)),
    color="C0",
    alpha=0.4,
    edgecolor=None,
)

plt.title("Western US")
plt.ylabel("Live AGB (MMT C)")
ax.spines[["right", "top"]].set_visible(False)
plt.legend(frameon=False)
ax.set_xlim([2005, 2023])
plt.axvspan(2005, 2015, color="lightgray", alpha=0.3)

# b) live biomass changes

In [ ]:
slope = xr.open_zarr(dir_info.dir_processed + "data_on_ref_grid/1000m/aspect.zarr")
crs = slope["spatial_ref"].crs_wkt

In [ ]:
inputs = xr.open_dataset(dir_info.dir_processed + "data_on_ref_grid/1000m/all_variables.nc")

In [ ]:
dir_data = dir_info.dir_model_output[:-1] + "_processed/"

In [ ]:
slope = xr.open_zarr(dir_info.dir_processed + "data_on_ref_grid/1000m/aspect.zarr")
crs = slope["spatial_ref"].crs_wkt
western_states = maps.SHP_WESTERN.to_crs(crs)

In [ ]:
def get_filtered_output(
    year,
    fdir=dir_data,
    ftype="nc",
    vartype="predicted_biomass_filtered_",
):
    fname = fdir + vartype + str(year)
    if ftype == "zarr":
        ds = xr.open_zarr(fname + ".zarr")
    elif ftype == "nc":
        ds = xr.load_dataset(fname + "_0000.nc")
    return ds


ds_end = get_filtered_output(year=2022)
ds_start = get_filtered_output(year=2005)
delta_biomass = ds_end["predicted_biomass"] - ds_start["predicted_biomass"]

We used the piecewise linear regression method to determine the turning point of total live BC across the northern ecosystems72. The Mann-Kendall nonparametric trend test was then used to calculate the temporal trends in total live BC after the turning point, while the Sen’s slope statistics were used to determine confidence intervals.

In [ ]:
inputs = xr.open_dataset(dir_info.dir_processed + "data_on_ref_grid/1000m/all_variables.nc")
inputs = inputs.rio.write_crs(crs)

In [ ]:
burned = inputs["years_after_fire"].sel(year=2022) <= 17

In [ ]:
burned_clipped = burned.rio.clip(western_states.geometry)

In [ ]:
biomass_start_clipped = (
    ds_start["predicted_biomass"].rio.write_crs(crs).rio.clip(western_states.geometry)
)

In [ ]:
delta_biomass = delta_biomass.rio.write_crs(crs)

In [ ]:
delta_biomass_clipped = delta_biomass.rio.clip(western_states.geometry)

In [ ]:
maps.plot_map(
    delta_biomass_clipped,  # .mean(),
    shp=western_states,
    latlon=False,
    title="Change in Live Aboveground Biomass from 2005-2020",
    cbar_label=r"$\Delta$ Live AGB (Mg/ha)",
    cmap=plt.cm.BrBG,
    clims=[-60, 60],
    savefig=None,
)
# plt.savefig('Figure2.pdf')

# Save data used for figure generation

In [ ]:
# Save data used for figure generation
import os

import pandas as pd

save_dir = "figure_data/figure_2/"
os.makedirs(save_dir, exist_ok=True)

# Panel a: biomass time series — saved as CSV
df_biomass = pd.DataFrame(
    {
        "year": biomass_mean_west.index,
        "biomass_mean": biomass_mean_west.values,
        "biomass_min": biomass_min_west.values,
        "biomass_max": biomass_max_west.values,
    }
)
df_biomass.to_csv(save_dir + "biomass_timeseries.csv", index=False)

# Panel b: spatial delta biomass map — saved as NetCDF
delta_biomass_clipped.to_dataset(name="delta_biomass_clipped").to_netcdf(
    save_dir + "delta_biomass_clipped.nc"
)

# CRS string (needed to reproject shapefile in figure notebook)
with open(save_dir + "target_crs.txt", "w") as f:
    f.write(crs)

print(f"Saved all figure data to {save_dir}")